Đọc dữ liệu

In [1]:

# CELL 2: IMPORT THU VIEN

import numpy as np  # Thu vien tinh toan so hoc
import pandas as pd  # Thu vien xu ly du lieu

from pathlib import Path  # Ho tro duong dan tep

from sklearn.model_selection import train_test_split  # Chia Train/Test
from sklearn.model_selection import GridSearchCV  # Tim kiem sieu tham so
from sklearn.model_selection import KFold  # Chia fold trong GridSearch

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa du lieu phan loai
from sklearn.compose import ColumnTransformer  # Ap dung ma hoa cho cac cot
from sklearn.pipeline import Pipeline  # Ket hop ma hoa va mo hinh

from sklearn.ensemble import RandomForestRegressor  # Mo hinh Random Forest

from sklearn.metrics import mean_squared_error  # Tinh MSE
from sklearn.metrics import mean_absolute_error  # Tinh MAE
from sklearn.metrics import r2_score  # Tinh R2

print("Da import thu vien thanh cong!")

# CELL 4: DOC DU LIEU

# Duong dan khi notebook chay tu experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Duong dan du phong khi chay tu thu muc goc
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Bao loi neu khong tim thay file
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file CSV")

# Doc file CSV, dung cot dau tien lam chi so
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan du lieu
print("Duong dan:", file_path.resolve())

# In kich thuoc du lieu ban dau
print("Kich thuoc:", diamonds.shape)

# In so gia tri thieu
print("Gia tri thieu:", diamonds.isnull().sum().sum())

# In so dong trung lap
print("Dong trung lap:", diamonds.duplicated().sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import thu vien thanh cong!
Duong dan: D:\DiamondPrediction\data\raw\Diamonds Prices2022.csv
Kich thuoc: (53943, 10)
Gia tri thieu: 0
Dong trung lap: 149


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

# CELL 6: TIEN XU LY DU LIEU

# Tao ban sao cua du lieu goc
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Loai bo cac dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xu ly gia tri thieu
after_missing = len(df)

# Chi giu cac dong co x, y, z lon hon 0
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Luu so dong sau khi loc kich thuoc
after_dimensions = len(df)

# Xoa cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong
df = df.reset_index(drop=True)

# In so dong bi loai do thieu du lieu
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong bi loai do kich thuoc
print("Dong kich thuoc khong hop le bi loai:",
      after_missing - after_dimensions)

# In so dong trung lap bi loai
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong sau khi xu ly
print("So dong sau tien xu ly:", len(df))

# Hien thi 5 dong dau tien
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le bi loai: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Chia tập dữ liệu

In [3]:

# CELL 8: CHIA DU LIEU HOLD-OUT

# Lay cac cot dac trung, bo cot price
X = df.drop(columns=["price"])

# Lay cot price lam bien muc tieu
y = df["price"]

# Chia du lieu thanh Train va Test
X_train, X_test, y_train, y_test = train_test_split(
    X,  # Cac thuoc tinh dau vao
    y,  # Gia kim cuong
    test_size=0.2,  # 20% du lieu lam Test
    random_state=42  # Co dinh cach chia
)

# In so mau Train
print("So mau Train:", len(X_train))

# In so mau Test
print("So mau Test:", len(X_test))


So mau Train: 43020
So mau Test: 10755



## 5. Xây dựng Pipeline

Sử dụng OrdinalEncoder để mã hóa ba thuộc tính:
- color
- clarity
- cut

Kết hợp bộ mã hóa và Random Forest bằng Pipeline.

Pipeline giúp bộ mã hóa được fit riêng trong từng fold của GridSearchCV, tránh sử dụng thông tin từ fold đánh giá.

Các thuộc tính số được giữ nguyên.


In [4]:

# CELL 10: TAO PIPELINE RANDOM FOREST

# Khai bao thu tu chat luong mau sac
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Khai bao thu tu do tinh khiet
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Khai bao thu tu chat luong giac cat
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao bo ma hoa voi thu tu co dinh
encoder = OrdinalEncoder(
    categories=[
        color_order,  # Thu tu cot color
        clarity_order,  # Thu tu cot clarity
        cut_order  # Thu tu cot cut
    ]
)

# Tao bo tien xu ly cho cac cot du lieu
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", encoder, categorical_cols)  # Ma hoa 3 cot
    ],
    remainder="passthrough"  # Giu nguyen cac cot so
)

# Khoi tao Random Forest
rf_model = RandomForestRegressor(
    random_state=42,  # Co dinh tinh ngau nhien
    n_jobs=-1  # Su dung cac loi CPU kha dung
)

# Ket hop tien xu ly va Random Forest
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # Buoc ma hoa
        ("model", rf_model)  # Buoc huan luyen
    ]
)

# In thong bao
print("Da tao Pipeline thanh cong!")


Da tao Pipeline thanh cong!



## 6. Tìm kiếm siêu tham số bằng GridSearchCV

Mức 3 sử dụng phạm vi tìm kiếm nhỏ:

- n_estimators: [100, 150]
- max_depth: [None, 20]
- min_samples_leaf: [1, 2]

Tổng cộng có 8 cấu hình.

Mỗi cấu hình được đánh giá bằng 3-Fold Cross-validation trên tập Train.

Sử dụng RMSE trung bình làm tiêu chí lựa chọn.

Cấu hình có RMSE trung bình thấp nhất được chọn để đánh giá trên tập Test.


In [5]:

# CELL 12: TIM KIEM SIEU THAM SO

# Khai bao pham vi tim kiem sieu tham so
param_grid = {
    "model__n_estimators": [100, 150],  # So cay trong rung
    "model__max_depth": [None, 20],  # Do sau toi da
    "model__min_samples_leaf": [1, 2]  # So mau toi thieu tai nut la
}

# Khoi tao bo chia 3-Fold cho tap Train
cv = KFold(
    n_splits=3,  # Chia Train thanh 3 fold
    shuffle=True,  # Tron ngau nhien truoc khi chia
    random_state=42  # Co dinh cach chia
)

# Khoi tao GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,  # Pipeline can toi uu
    param_grid=param_grid,  # Danh sach sieu tham so
    scoring="neg_root_mean_squared_error",  # Chon theo RMSE
    cv=cv,  # Su dung 3-Fold
    n_jobs=1,  # Tranh chay song song long nhau
    refit=True,  # Huan luyen lai cau hinh tot nhat
    verbose=1,  # In tien do huan luyen
    return_train_score=True  # Luu diem Train de tham khao
)

# Chay tim kiem tren tap Train
grid_search.fit(X_train, y_train)

# In cau hinh sieu tham so tot nhat
print("Sieu tham so tot nhat:")
print(grid_search.best_params_)

# In RMSE trung binh CV cua cau hinh tot nhat
print(f"RMSE CV tot nhat: {-grid_search.best_score_:.4f}")


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Sieu tham so tot nhat:
{'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 150}
RMSE CV tot nhat: 555.0171



## 7. Kết quả tìm kiếm siêu tham số

Hiển thị các cấu hình đã thử và RMSE trung bình tương ứng.

Sắp xếp các cấu hình theo RMSE từ thấp đến cao.

Cấu hình có RMSE nhỏ nhất được chọn làm mô hình tốt nhất.


In [6]:

# CELL 14: BANG KET QUA GRID SEARCH

# Chuyen ket qua GridSearchCV thanh DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# Chon cac cot can thiet
search_results = cv_results[
    [
        "param_model__n_estimators",  # So cay
        "param_model__max_depth",  # Do sau toi da
        "param_model__min_samples_leaf",  # So mau tai nut la
        "mean_test_score",  # Diem CV trung binh
        "std_test_score",  # Do lech chuan diem CV
        "rank_test_score"  # Thu hang cau hinh
    ]
].copy()

# Doi diem am thanh RMSE duong
search_results["RMSE_CV"] = -search_results["mean_test_score"]

# Lay do lech chuan cua RMSE
search_results["RMSE_STD"] = search_results["std_test_score"]

# Sap xep theo RMSE tu thap den cao
search_results = search_results.sort_values("RMSE_CV")

# Hien thi bang ket qua
display(search_results.round(4))


,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,mean_test_score,std_test_score,rank_test_score,RMSE_CV,RMSE_STD
3,150,None,2,-555.0171,2.0163,1,555.0171,2.0163
7,150,20,2,-555.0563,2.1526,2,555.0563,2.1526
2,100,None,2,-556.2593,1.6280,3,556.2593,1.6280
6,100,20,2,-556.4642,1.8003,4,556.4642,1.8003
5,150,20,1,-556.4929,2.1737,5,556.4929,2.1737
1,150,None,1,-557.0582,1.8230,6,557.0582,1.8230
4,100,20,1,-557.8741,1.7526,7,557.8741,1.7526
0,100,None,1,-558.4761,1.4556,8,558.4761,1.4556



## 8. Đánh giá mô hình tốt nhất trên Test

Lấy mô hình tốt nhất từ GridSearchCV.

Mô hình này đã được huấn luyện lại trên toàn bộ tập Train.

Dự đoán trên tập Test được giữ riêng.

Tính MSE, RMSE, MAE và R² để đánh giá hiệu suất cuối cùng.


In [7]:

# CELL 16: DANH GIA MO HINH TOT NHAT

# Lay Pipeline tot nhat da duoc huan luyen
best_model = grid_search.best_estimator_

# Du doan gia kim cuong tren tap Test
y_pred = best_model.predict(X_test)

# Tinh sai so binh phuong trung binh
mse = mean_squared_error(y_test, y_pred)

# Tinh can bac hai cua MSE
rmse = np.sqrt(mse)

# Tinh sai so tuyet doi trung binh
mae = mean_absolute_error(y_test, y_pred)

# Tinh he so xac dinh R2
r2 = r2_score(y_test, y_pred)

# In ten thuc nghiem
print("MUC 3 - RANDOM FOREST HOLD-OUT 80/20")

# In duong phan cach
print("-" * 45)

# In RMSE trung binh CV tot nhat
print(f"RMSE CV tot nhat: {-grid_search.best_score_:.4f}")

# In MSE tren Test
print(f"MSE Test : {mse:.4f}")

# In RMSE tren Test
print(f"RMSE Test: {rmse:.4f}")

# In MAE tren Test
print(f"MAE Test : {mae:.4f}")

# In R2 tren Test
print(f"R2 Test  : {r2:.4f}")

# In sieu tham so tot nhat
print("Sieu tham so tot nhat:", grid_search.best_params_)


MUC 3 - RANDOM FOREST HOLD-OUT 80/20
---------------------------------------------
RMSE CV tot nhat: 555.0171
MSE Test : 270770.8141
RMSE Test: 520.3564
MAE Test : 264.5281
R2 Test  : 0.9829
Sieu tham so tot nhat: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 150}


So sánh hai mức

In [8]:

# CELL 18: SO SANH MUC 2 VA MUC 3

# Luu ket qua Hold-out muc 2 da chay
level2_rmse = 522.1064  # RMSE muc 2
level2_mae = 265.3777  # MAE muc 2
level2_r2 = 0.9828  # R2 muc 2

# Tao bang so sanh
comparison = pd.DataFrame({
    "Chi so": ["RMSE", "MAE", "R2"],
    "Muc 2": [level2_rmse, level2_mae, level2_r2],
    "Muc 3": [rmse, mae, r2]
})

# Tinh chenh lech Muc 3 tru Muc 2
comparison["Chenh lech"] = comparison["Muc 3"] - comparison["Muc 2"]

# Hien thi bang so sanh
display(comparison.round(4))

# Tinh phan tram thay doi RMSE
rmse_change = (rmse - level2_rmse) / level2_rmse * 100

# In phan tram thay doi RMSE
print(f"Thay doi RMSE: {rmse_change:+.2f}%")

# Kiem tra RMSE co cai thien hay khong
if rmse < level2_rmse:
    print("RMSE Muc 3 thap hon Muc 2.")

elif rmse > level2_rmse:
    print("RMSE Muc 3 cao hon Muc 2.")

else:
    print("RMSE hai muc bang nhau.")


,Chi so,Muc 2,Muc 3,Chenh lech
0,RMSE,522.1064,520.3564,-1.7500
1,MAE,265.3777,264.5281,-0.8496
2,R2,0.9828,0.9829,0.0001


Thay doi RMSE: -0.34%
RMSE Muc 3 thap hon Muc 2.
